# Claude Code — Skills

---

## What Skills Are

Reusable markdown files that teach Claude Code how to handle specific tasks
automatically — without repeating the same instructions every time.

> If you find yourself explaining the same thing to Claude repeatedly,
> that's a skill waiting to be written.

---

## Skill File Structure

Each skill lives in a `SKILL.md` file with frontmatter:
```markdown
---
name: pr-review
description: Reviews pull requests for code quality. Use when reviewing PRs or checking code changes.
---

## PR Review Checklist

- Check for type safety
- Verify error handling
- Review naming conventions
...
```

- **`name`** — identifies the skill
- **`description`** — how Claude decides whether to activate the skill
- **Body** — the actual instructions Claude follows

---

## Where Skills Live

| Location | Who it applies to |
|---|---|
| `~/.claude/skills/` | You — follows you across all projects |
| `.claude/skills/` (in repo) | Everyone — committed to version control, shared with the team |
| `C:/Users/<user>/.claude/skills/` | Windows personal skills |

---

## How Skills Activate

Claude loads only the name and description of each skill initially.
When you make a request, Claude compares it against all skill descriptions
and automatically loads matching ones.
```
You: "Review this PR"
        |
        v
Claude scans skill descriptions
        |
        v
Matches: pr-review skill
        |
        v
Skill instructions loaded into context
        |
        v
Claude reviews the PR using your checklist
```

You will see the skill load in the terminal when it activates.

---

## Skills vs CLAUDE.md vs Slash Commands

| | CLAUDE.md | Skills | Slash Commands |
|---|---|---|---|
| **When it loads** | Every conversation | On demand when matched | Only when explicitly typed |
| **Activation** | Always | Automatic | Manual |
| **Context cost** | Always in context | Only when relevant | Only when invoked |
| **Best for** | Project-wide constants | Task-specific knowledge | One-off explicit workflows |

---

## When to Use Skills

- Code review standards your team follows
- Commit message formats you prefer
- Brand guidelines for your organisation
- Documentation templates for specific doc types
- Debugging checklists for particular frameworks

---

## Key Takeaway

> Skills are automatic and task-specific — unlike CLAUDE.md (always loaded)
> or slash commands (manually invoked).  
> Write a skill once, and Claude applies it every time the situation matches.  
> Personal skills follow you; project skills follow the repo.

# Claude Code — Building a Skill

---

## Skill File Structure

A skill is a **directory** containing a `SKILL.md` file:
```
~/.claude/skills/pr-description/
    └── SKILL.md
```
```markdown
---
name: pr-description
description: Writes pull request descriptions. Use when creating a PR,
             writing a PR, or when the user asks to summarize changes
             for a pull request.
---

When writing a PR description:

1. Run `git diff main...HEAD` to see all changes on this branch
2. Write a description following this format:

## What
One sentence explaining what this PR does.

## Why
Brief context on why this change is needed.

## Changes
- Bullet points of specific changes made
- Group related changes together
- Mention any files deleted or renamed
```

- **`name`** — identifies the skill (directory name should match)
- **`description`** — the matching criteria Claude uses
- **Body** — instructions Claude follows when the skill activates

---

## How Skill Matching Works
```
Claude Code starts
        |
        v
Scans 4 locations, loads only name + description (not full content)
        |
        v
You send: "write a PR description for my changes"
        |
        v
Claude semantically matches request against all skill descriptions
        |
        v
Match found → Claude asks you to confirm loading the skill
        |
        v
You confirm → full SKILL.md loaded into context
        |
        v
Claude follows the instructions
```

> Confirmation step keeps you aware of what context Claude is pulling in.

---

## Skill Priority (Highest to Lowest)

| Priority | Source | Location |
|---|---|---|
| 1 | **Enterprise** | Managed/org settings |
| 2 | **Personal** | `~/.claude/skills/` |
| 3 | **Project** | `.claude/skills/` in repo |
| 4 | **Plugins** | Installed plugins |

If a cloned repo has a skill with the same name as your personal skill,
your personal version wins. Enterprise always wins over everything.

**Tip:** Use descriptive names to avoid conflicts — `frontend-review`
rather than `review`.

---

## Creating, Updating, Removing
```bash
# Create
mkdir -p ~/.claude/skills/pr-description
# then create SKILL.md inside it

# Update
# Edit the SKILL.md file directly

# Remove
rm -rf ~/.claude/skills/pr-description
```

> Always **restart Claude Code** after creating, editing, or deleting skills.

---

## Skills vs CLAUDE.md — Loading Comparison
```
CLAUDE.md      → full content loaded into every conversation (always in context)
Skills         → only name + description loaded at startup
                 full content loaded only when matched + confirmed
```

Skills are context-efficient — your PR review checklist is not consuming
context while you debug unrelated code.

---

## Key Takeaway

> A skill = a directory + a `SKILL.md` with frontmatter name/description + instructions.  
> Claude matches semantically — overlapping intent triggers the skill.  
> Priority order: Enterprise → Personal → Project → Plugins.  
> Restart Claude Code after any skill changes.

# Claude Code — Advanced Skill Techniques

---

## All Metadata Fields

| Field | Required | Detail |
|---|---|---|
| `name` | Yes | Lowercase, numbers, hyphens only. Max 64 chars. Match directory name. |
| `description` | Yes | Max 1,024 chars. Most important field — Claude uses this for matching. |
| `allowed-tools` | No | Restricts which tools Claude can use when the skill is active |
| `model` | No | Specifies which Claude model to use for this skill |

---

## Writing Effective Descriptions

A good description answers two questions:
```
1. What does the skill do?
2. When should Claude use it?
```

**Vague (won't trigger reliably):**
```yaml
description: Helps with docs.
```

**Specific (triggers reliably):**
```yaml
description: Writes and updates technical documentation. Use when creating
             API docs, writing READMEs, updating changelogs, or when the
             user asks to document a function, module, or feature.
```

> If your skill isn't triggering when expected, add more keywords that
> match how you actually phrase your requests.

---

## Restricting Tools with `allowed-tools`

Limit what Claude can do when the skill is active — useful for read-only
or security-sensitive workflows:
```yaml
---
name: codebase-onboarding
description: Helps new developers understand how the system works.
allowed-tools: Read, Grep, Glob, Bash
model: sonnet
---
```

When this skill is active, Claude can only use `Read`, `Grep`, `Glob`,
and `Bash` — no writing or editing files.

If `allowed-tools` is omitted, the skill does not restrict anything.

---

## Progressive Disclosure — Structuring Larger Skills

**Problem:** Cramming everything into one large `SKILL.md`:
- Consumes too much context window space
- Difficult to maintain

**Solution:** Keep `SKILL.md` under 500 lines. Put supporting material
in separate files that Claude reads **only when needed**.

### Recommended Directory Structure
```
.claude/skills/system-design/
    ├── SKILL.md                  ← essential instructions + links
    ├── references/
    │   └── architecture-guide.md ← loaded only when asked about architecture
    ├── scripts/
    │   └── validate-env.sh       ← executed, not read into context
    └── assets/
        └── component-template.md
```

### Linking to Supporting Files in `SKILL.md`
```markdown
---
name: system-design
description: Guides system design decisions. Use when the user asks about
             architecture, component placement, or system structure.
---

For component placement questions, refer to the component guidelines below.

For system architecture questions, read references/architecture-guide.md
only when the user explicitly asks about system design or architecture.
```

Claude loads `architecture-guide.md` only when relevant — not for every request.

---

## Using Scripts Efficiently

Scripts **execute** without loading their contents into context.
Only the **output** consumes tokens.
```markdown
# In SKILL.md — tell Claude to RUN the script, not read it

Before starting, run scripts/validate-env.sh to check the environment.
Do not read the script file — execute it and use the output.
```

**Good for:**
- Environment validation
- Consistent data transformations
- Operations more reliable as tested code than generated code

---

## Summary — Skill Size Guidelines

| Content | Where it goes |
|---|---|
| Core instructions | `SKILL.md` (keep under 500 lines) |
| Detailed reference docs | `references/` — linked, loaded on demand |
| Executable utilities | `scripts/` — run, not read |
| Templates, images | `assets/` — referenced when needed |

---

## Key Takeaway

> `name` and `description` are required — be explicit in descriptions
> so Claude matches reliably.  
> `allowed-tools` adds guardrails for read-only or sensitive workflows.  
> Progressive disclosure keeps context efficient — SKILL.md under 500 lines,
> supporting detail in separate files loaded only when needed.  
> Scripts run without consuming context — only their output counts.

# Claude Code — Choosing the Right Customisation Feature

---

## The Five Options

| Feature | When it loads | Triggered by |
|---|---|---|
| **CLAUDE.md** | Every conversation, always | Automatic |
| **Skills** | On demand when matched | Your request (semantic match) |
| **Subagents** | In a separate isolated context | Explicit delegation |
| **Hooks** | On specific events | File saves, tool calls, etc. |
| **MCP servers** | When connected | Tool calls to external services |

---

## CLAUDE.md vs Skills

| | CLAUDE.md | Skills |
|---|---|---|
| **Loads** | Every conversation | Only when request matches |
| **Context cost** | Always in context | Only when activated |
| **Best for** | Always-on project standards | Task-specific expertise |

**Use CLAUDE.md for:**
- Project-wide standards that always apply
- Hard constraints — "never modify the database schema"
- Framework preferences and coding style

**Use Skills for:**
- Task-specific expertise (PR reviews, commit messages)
- Knowledge only relevant sometimes
- Detailed procedures that would clutter every conversation

---

## Skills vs Subagents

| | Skills | Subagents |
|---|---|---|
| **Context** | Joins the current conversation | Runs in an isolated context |
| **Scope** | Enhances Claude's knowledge | Delegates a task independently |
| **Results** | Applied throughout conversation | Returned to main conversation |

**Use Skills when:** You want to add knowledge to the current task.
**Use Subagents when:** You want to delegate work to a separate execution context
with potentially different tool access.

---

## Skills vs Hooks

| | Skills | Hooks |
|---|---|---|
| **Triggered by** | What you're asking (request-driven) | Events (file save, tool call) |
| **Purpose** | Informs Claude's reasoning | Automated side effects |

**Use Hooks for:**
- Running a linter every time Claude saves a file
- Validation before specific tool calls
- Automated operations triggered by Claude's actions

**Use Skills for:**
- Knowledge that informs how Claude handles requests
- Guidelines that affect Claude's reasoning process

---

## A Typical Full Setup
```
CLAUDE.md          → always-on project standards (TypeScript strict mode, etc.)
Skills             → task-specific expertise loaded on demand (PR review, docs)
Hooks              → automated operations on events (format on save, lint on edit)
Subagents          → isolated execution for delegated work
MCP servers        → external tools and integrations (GitHub, Slack, databases)
```

These are complementary — use multiple at once rather than forcing everything
into one approach.

---

## Quick Decision Guide
```
Should this always be in context?
    Yes → CLAUDE.md

Is this task-specific knowledge that should load automatically?
    Yes → Skill

Should this run every time a specific event happens?
    Yes → Hook

Does this need its own isolated execution context?
    Yes → Subagent

Does this connect to an external service or tool?
    Yes → MCP server
```

---

## Key Takeaway

> Each feature handles its own specialty — don't force everything into one.  
> Skills = automatic task-specific expertise.  
> CLAUDE.md = always-on instructions.  
> Hooks = event-driven automation.  
> Subagents = isolated delegated work.  
> MCP = external tools and integrations.

# Claude Code — Sharing & Distributing Skills

---

## Three Distribution Methods

### 1. Commit to Repository (`.claude/skills/`)
```
.claude/
    skills/
        pr-review/
            SKILL.md
        commit-message/
            SKILL.md
```

- Anyone who clones the repo gets the skills automatically
- Updates pushed via normal Git workflows
- **Best for:** Team coding standards, project-specific workflows,
  skills that reference your codebase structure

---

### 2. Plugins — Distribute via Marketplace

Create a `skills/` directory in your plugin project following the same
file structure as `.claude/`. Publish to a marketplace — other users
discover and install it.

- **Best for:** Skills not tied to a specific project, community sharing,
  skills useful beyond your immediate team

---

### 3. Enterprise Managed Settings

Administrators deploy skills organisation-wide. Enterprise skills have the
**highest priority** — override personal, project, and plugin skills with
the same name.
```json
"strictKnownMarketplaces": [
  { "source": "github", "repo": "acme-corp/approved-plugins" },
  { "source": "npm", "package": "@acme-corp/compliance-plugins" }
]
```

- **Best for:** Mandatory standards, security requirements, compliance
  workflows — anything that **must** be consistent across the organisation

---

## Distribution Method Comparison

| Method | Scope | Priority | Best for |
|---|---|---|---|
| Repository (`.claude/skills/`) | Project team | 3rd | Team standards, project workflows |
| Plugins | Community | 4th (lowest) | Cross-project, community sharing |
| Enterprise managed | Organisation | 1st (highest) | Mandatory org-wide standards |
| Personal (`~/.claude/skills/`) | You only | 2nd | Personal preferences |

---

## Skills and Subagents — Important Gotcha

> Subagents do **not** automatically inherit skills.
> They start with a clean, fresh context.

| Agent type | Can use skills? |
|---|---|
| Built-in agents (Explorer, Plan, Verify) | No — cannot access skills at all |
| Custom subagents in `.claude/agents/` | Yes — but only if explicitly listed |

---

## Adding Skills to a Custom Subagent

Create or edit an agent file in `.claude/agents/`:
```yaml
---
name: frontend-security-accessibility-reviewer
description: "Use this agent when you need to review frontend code for
              accessibility and security issues."
tools: Bash, Glob, Grep, Read, WebFetch, WebSearch
model: sonnet
color: blue
skills: accessibility-audit, performance-check
---
```

- Skills listed in `skills:` are loaded when the subagent **starts**
  (not on demand like in the main conversation)
- The skills must exist in `.claude/skills/`
- Use `/agents` command in Claude Code to create one interactively

---

## When Custom Subagents with Skills Work Best

- Isolated task delegation with specific expertise
- Different subagents for different domains (frontend reviewer vs backend reviewer)
- Enforcing standards in delegated work without relying on prompts

---

## Key Takeaway

> Commit to repo for team sharing, plugins for community, enterprise for org mandates.  
> Subagents do not inherit skills automatically —
> explicitly list them in the agent's `skills:` frontmatter.  
> Built-in agents (Explorer, Plan, Verify) cannot use skills at all —
> only custom agents in `.claude/agents/` can.

    # Claude Code — Skills Troubleshooting

---

## Start Here — Skills Validator

Run the validator first before debugging anything else:
```bash
# Install via uv (recommended)
uv tool install agent-skills-verifier

# Run from your skill directory
agent-skills-verifier
```

Catches structural problems before you spend time chasing other issues.

---

## Problem 1 — Skill Doesn't Trigger

**Cause:** Description doesn't semantically overlap with how you phrase requests.

**Fix:**
- Compare your description against how you actually phrase requests
- Add trigger phrases users would naturally say
- Test with variations: "help me profile this", "why is this slow?", "make this faster"
- If any variation fails → add those keywords to the description
```yaml
# Too vague — won't trigger reliably
description: Helps with performance.

# Specific — triggers reliably
description: Analyzes and improves code performance. Use when profiling code,
             investigating slow functions, or when the user asks why something
             is slow, how to make it faster, or to optimize performance.
```

---

## Problem 2 — Skill Doesn't Load

**Check these structural requirements:**
```
✓ SKILL.md is inside a named directory — NOT at the skills root
✓ File name is exactly SKILL.md (all caps SKILL, lowercase md)
✓ YAML frontmatter syntax is valid
```
```bash
# Correct structure
.claude/skills/pr-review/SKILL.md   ✓

# Wrong — file at root
.claude/skills/SKILL.md             ✗

# Wrong — file name
.claude/skills/pr-review/skill.md   ✗
.claude/skills/pr-review/Skill.MD   ✗
```
```bash
# Run with debug flag to see loading errors
claude --debug
```

Look for messages mentioning your skill name in the debug output.

---

## Problem 3 — Wrong Skill Gets Used

**Cause:** Descriptions are too similar — Claude can't distinguish between skills.

**Fix:** Make descriptions more specific and distinct from each other.
```yaml
# Too similar — will conflict
description: Reviews code for quality.
description: Reviews code for problems.

# Distinct — won't conflict
description: Reviews frontend code for accessibility and CSS standards.
description: Reviews backend code for security vulnerabilities and SQL injection.
```

---

## Problem 4 — Skill Priority Conflict

Your personal skill is being ignored because a higher-priority skill has the same name.

**Priority order:** Enterprise → Personal → Project → Plugins
```
Enterprise "code-review" skill exists
→ Your personal "code-review" skill is always overridden
```

**Options:**
- Rename your skill to something more specific (`my-code-review`, `backend-review`)
- Talk to your admin about the enterprise skill

---

## Problem 5 — Plugin Skills Not Appearing
```
1. Clear the plugin cache
2. Restart Claude Code
3. Reinstall the plugin
```

If skills still don't appear → validate the plugin's structure with the validator tool.

---

## Problem 6 — Runtime Errors (Skill Loads but Fails)

| Cause | Fix |
|---|---|
| Missing dependencies | Install required packages; add dependency info to skill description |
| Script permission denied | `chmod +x scripts/your-script.sh` |
| Path separator issues | Use forward slashes everywhere, including on Windows |

---

## Quick Troubleshooting Checklist

| Symptom | Fix |
|---|---|
| Not triggering | Improve description, add trigger phrases |
| Not loading | Check path, file name (`SKILL.md`), YAML syntax |
| Wrong skill used | Make descriptions more distinct |
| Being shadowed | Check priority hierarchy, rename your skill |
| Plugin skills missing | Clear cache, restart, reinstall |
| Runtime failure | Check dependencies, `chmod +x`, use `/` not `\` |

---

## Key Takeaway

> Most skill problems fall into one of six categories.  
> Start with the validator — it catches structural issues instantly.  
> Triggering problems are almost always a description issue — add keywords
> that match how you actually phrase requests.